In [1]:
import os
os.chdir('/home/cevdetkopuz/mercari-price-prediction')

import pandas as pd
import numpy as np
import gc
import re
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

print("Hazır")

Hazır


In [2]:
df = pd.read_csv('data/raw/train.tsv', sep='\t')
df = df[df['price'] > 0].reset_index(drop=True)
y = np.log1p(df['price']).values.astype(np.float32)

df['brand_name'].fillna('missing', inplace=True)
df['category_name'].fillna('missing/missing/missing', inplace=True)
df['item_description'].fillna('No description yet', inplace=True)

def split_category(cat):
    parts = str(cat).split('/')
    while len(parts) < 3: parts.append('missing')
    return parts[:3]

cs = df['category_name'].apply(split_category)
df['cat_main'] = cs.apply(lambda x: x[0])
df['cat_sub1'] = cs.apply(lambda x: x[1])
df['cat_sub2'] = cs.apply(lambda x: x[2])

top_brands = set(df[df['brand_name'] != 'missing']['brand_name'].value_counts().head(500).index)
df['brand_name'] = df['brand_name'].apply(lambda x: x if x in top_brands else 'other')

le_cat_main = LabelEncoder().fit(df['cat_main'])
le_cat_sub1 = LabelEncoder().fit(df['cat_sub1'])
le_cat_sub2 = LabelEncoder().fit(df['cat_sub2'])
le_brand = LabelEncoder().fit(df['brand_name'])

df['cat_main_enc'] = le_cat_main.transform(df['cat_main'])
df['cat_sub1_enc'] = le_cat_sub1.transform(df['cat_sub1'])
df['cat_sub2_enc'] = le_cat_sub2.transform(df['cat_sub2'])
df['brand_enc'] = le_brand.transform(df['brand_name'])
df['desc_word_count'] = df['item_description'].str.split().str.len()
df['name_len'] = df['name'].str.len()
df['has_description'] = (df['item_description'] != 'No description yet').astype(int)

# Birleşik metin (cross-field sinyal)
df['combined_text'] = df['name'] + ' ' + df['brand_name'] + ' ' + df['cat_main'] + ' ' + df['cat_sub2']

numeric_features = ['item_condition_id', 'shipping', 'cat_main_enc',
                    'cat_sub1_enc', 'cat_sub2_enc', 'brand_enc',
                    'desc_word_count', 'name_len', 'has_description']

print(f"Veri: {len(df):,} satır")

Veri: 1,481,661 satır


In [3]:
from scipy.sparse import save_npz, load_npz
os.makedirs('data/processed/sparse', exist_ok=True)

print("TF-IDF 1/4: name word..."); t0 = time.time()
tfidf_name_word = TfidfVectorizer(
    max_features=40000, ngram_range=(1, 2),
    strip_accents='unicode', token_pattern=r'\w{1,}', dtype=np.float32)
X = tfidf_name_word.fit_transform(df['name'])
save_npz('data/processed/sparse/name_word.npz', X)
joblib.dump(tfidf_name_word, 'models/tfidf_name_word_v2.joblib')
del X, tfidf_name_word; gc.collect()
print(f"  Kaydedildi, {time.time()-t0:.0f}s")

print("TF-IDF 2/4: name char..."); t0 = time.time()
tfidf_name_char = TfidfVectorizer(
    max_features=30000, analyzer='char_wb', ngram_range=(3, 5),
    strip_accents='unicode', dtype=np.float32, min_df=5)
X = tfidf_name_char.fit_transform(df['name'])
save_npz('data/processed/sparse/name_char.npz', X)
joblib.dump(tfidf_name_char, 'models/tfidf_name_char_v2.joblib')
del X, tfidf_name_char; gc.collect()
print(f"  Kaydedildi, {time.time()-t0:.0f}s")

print("TF-IDF 3/4: desc word..."); t0 = time.time()
tfidf_desc_word = TfidfVectorizer(
    max_features=40000, ngram_range=(1, 2),
    strip_accents='unicode', token_pattern=r'\w{1,}', dtype=np.float32)
X = tfidf_desc_word.fit_transform(df['item_description'])
save_npz('data/processed/sparse/desc_word.npz', X)
joblib.dump(tfidf_desc_word, 'models/tfidf_desc_word_v2.joblib')
del X, tfidf_desc_word; gc.collect()
print(f"  Kaydedildi, {time.time()-t0:.0f}s")

print("TF-IDF 4/4: combined..."); t0 = time.time()
tfidf_combined = TfidfVectorizer(
    max_features=20000, ngram_range=(1, 2),
    strip_accents='unicode', token_pattern=r'\w{1,}', dtype=np.float32)
X = tfidf_combined.fit_transform(df['combined_text'])
save_npz('data/processed/sparse/combined.npz', X)
joblib.dump(tfidf_combined, 'models/tfidf_combined_v2.joblib')
del X, tfidf_combined; gc.collect()
print(f"  Kaydedildi, {time.time()-t0:.0f}s")

print("\nTüm TF-IDF matrisleri diske kaydedildi!")

TF-IDF 1/4: name word...
  Kaydedildi, 37s
TF-IDF 2/4: name char...
  Kaydedildi, 107s
TF-IDF 3/4: desc word...
  Kaydedildi, 139s
TF-IDF 4/4: combined...
  Kaydedildi, 51s

Tüm TF-IDF matrisleri diske kaydedildi!


In [4]:
from scipy.sparse import load_npz

print("Matrisler diskten yükleniyor...")
X_nw = load_npz('data/processed/sparse/name_word.npz')
X_nc = load_npz('data/processed/sparse/name_char.npz')
X_dw = load_npz('data/processed/sparse/desc_word.npz')
X_cb = load_npz('data/processed/sparse/combined.npz')
X_num = csr_matrix(df[numeric_features].values, dtype=np.float32)

X = hstack([X_num, X_nw, X_nc, X_dw, X_cb]).tocsr()
del X_num, X_nw, X_nc, X_dw, X_cb; gc.collect()

print(f"Toplam: {X.shape[1]:,} feature")

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nRidge Alpha Tuning")
print("-" * 45)

best_score = float('inf')
best_alpha = None

for alpha in [1.0, 3.0, 5.0, 7.0, 10.0]:
    t0 = time.time()
    m = Ridge(alpha=alpha)
    m.fit(X_train, y_train)
    pred = np.maximum(m.predict(X_val), 0)
    score = np.sqrt(mean_squared_error(y_val, pred))
    
    marker = " <-- best" if score < best_score else ""
    if score < best_score:
        best_score = score
        best_alpha = alpha
        best_ridge = m
    
    del m; gc.collect()
    print(f"  alpha={alpha:<6} RMSLE={score:.4f} ({time.time()-t0:.0f}s){marker}")

print("-" * 45)
print(f"En iyi: alpha={best_alpha}, RMSLE={best_score:.4f}")
print(f"Önceki Ridge: 0.4733, İyileşme: {0.4733 - best_score:+.4f}")

# Kaydet
y_val_pred_ridge = np.maximum(best_ridge.predict(X_val), 0)
np.save('models/y_val_pred_ridge_v2.npy', y_val_pred_ridge)
np.save('models/y_val_v2.npy', y_val)
joblib.dump(best_ridge, 'models/ridge_v2.joblib')
print("Kaydedildi")

Matrisler diskten yükleniyor...
Toplam: 130,009 feature

Ridge Alpha Tuning
---------------------------------------------
  alpha=1.0    RMSLE=0.4476 (625s) <-- best
  alpha=3.0    RMSLE=0.4465 (590s) <-- best
  alpha=5.0    RMSLE=0.4471 (576s)
  alpha=7.0    RMSLE=0.4497 (449s)
  alpha=10.0   RMSLE=0.4513 (369s)
---------------------------------------------
En iyi: alpha=3.0, RMSLE=0.4465
Önceki Ridge: 0.4733, İyileşme: +0.0268
Kaydedildi


In [5]:
del best_ridge, X_train, X_val, X, y_val_pred_ridge
gc.collect()
print("Bellek temizlendi")

from sklearn.decomposition import TruncatedSVD

# Diskten yükle
X_nw = load_npz('data/processed/sparse/name_word.npz')
X_nc = load_npz('data/processed/sparse/name_char.npz')
X_dw = load_npz('data/processed/sparse/desc_word.npz')
X_cb = load_npz('data/processed/sparse/combined.npz')

X_all = hstack([X_nw, X_nc, X_dw, X_cb]).tocsr()
del X_nw, X_nc, X_dw, X_cb; gc.collect()

print("SVD (130K → 200)..."); t0 = time.time()
svd = TruncatedSVD(n_components=200, random_state=42, n_iter=5)
X_svd = svd.fit_transform(X_all).astype(np.float32)
print(f"  {time.time()-t0:.0f}s, varyans: {svd.explained_variance_ratio_.sum():.3f}")
del X_all; gc.collect()

X_numeric = df[numeric_features].values.astype(np.float32)
X_lgb = np.hstack([X_numeric, X_svd]).astype(np.float32)
del X_svd, X_numeric; gc.collect()

print(f"LightGBM matrisi: {X_lgb.shape}")

import lightgbm as lgb

X_tr, X_vl, y_tr, y_vl = train_test_split(X_lgb, y, test_size=0.2, random_state=42)

lgb_params = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.05, 'num_leaves': 127,
    'max_depth': -1, 'min_data_in_leaf': 100,
    'feature_fraction': 0.9, 'bagging_fraction': 0.9,
    'bagging_freq': 5, 'verbose': -1, 'n_jobs': -1, 'seed': 42,
}

print("\nLightGBM eğitiliyor...")
t0 = time.time()
tr_ds = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
vl_ds = lgb.Dataset(X_vl, y_vl, reference=tr_ds, free_raw_data=True)

lgb_model = lgb.train(
    lgb_params, tr_ds, num_boost_round=2000,
    valid_sets=[vl_ds], valid_names=['val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)],
)

print(f"\nEğitim: {time.time()-t0:.0f}s")
y_vl_pred_lgb = np.maximum(lgb_model.predict(X_vl), 0)
lgb_rmsle = np.sqrt(mean_squared_error(y_vl, y_vl_pred_lgb))
print(f"LightGBM v2 RMSLE: {lgb_rmsle:.4f}")

np.save('models/y_val_pred_lgb_v2.npy', y_vl_pred_lgb)
lgb_model.save_model('models/lgbm_v2.txt')
joblib.dump(svd, 'models/svd_v2.joblib')
print("Kaydedildi")

Bellek temizlendi
SVD (130K → 200)...
  334s, varyans: 0.243
LightGBM matrisi: (1481661, 209)

LightGBM eğitiliyor...
Training until validation scores don't improve for 50 rounds
[200]	val's rmse: 0.532308
[400]	val's rmse: 0.51271
[600]	val's rmse: 0.503865
[800]	val's rmse: 0.499554
[1000]	val's rmse: 0.496816
[1200]	val's rmse: 0.494416
[1400]	val's rmse: 0.492328
[1600]	val's rmse: 0.490694
[1800]	val's rmse: 0.489312
[2000]	val's rmse: 0.488061
Did not meet early stopping. Best iteration is:
[2000]	val's rmse: 0.488061

Eğitim: 576s
LightGBM v2 RMSLE: 0.4881
Kaydedildi


In [6]:
y_val = np.load('models/y_val_v2.npy')
y_ridge = np.load('models/y_val_pred_ridge_v2.npy')
y_lgb = np.load('models/y_val_pred_lgb_v2.npy')

ridge_solo = np.sqrt(mean_squared_error(y_val, y_ridge))
lgb_solo = np.sqrt(mean_squared_error(y_val, y_lgb))

best_ens = float('inf')
best_w = None
for w in np.arange(0.3, 0.8, 0.05):
    r = np.sqrt(mean_squared_error(y_val, w*y_ridge + (1-w)*y_lgb))
    if r < best_ens:
        best_ens = r
        best_w = round(w, 2)

print("=" * 55)
print("FINAL SKOR TABLOSU")
print("=" * 55)
print(f"  Ridge v1 (eski):          0.4733")
print(f"  Ridge v2 (yeni):          {ridge_solo:.4f}")
print(f"  LightGBM v1 (eski):       0.4880")
print(f"  LightGBM v2 (yeni):       {lgb_solo:.4f}")
print(f"  Ensemble v1 (eski):       0.4557")
print(f"  Ensemble v2 (yeni):       {best_ens:.4f}")
print(f"  Ağırlık: Ridge={best_w}, LGB={round(1-best_w,2)}")
print(f"=" * 55)
print(f"  Toplam iyileşme:          {0.4733 - best_ens:+.4f}")

FINAL SKOR TABLOSU
  Ridge v1 (eski):          0.4733
  Ridge v2 (yeni):          0.4465
  LightGBM v1 (eski):       0.4880
  LightGBM v2 (yeni):       0.4881
  Ensemble v1 (eski):       0.4557
  Ensemble v2 (yeni):       0.4396
  Ağırlık: Ridge=0.75, LGB=0.25
  Toplam iyileşme:          +0.0337


In [7]:
from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb

# SVD verisini yeniden oluştur (bellekte yoksa)
try:
    X_lgb.shape
    print("X_lgb bellekte mevcut")
except:
    print("SVD yeniden oluşturuluyor...")
    X_nw = load_npz('data/processed/sparse/name_word.npz')
    X_nc = load_npz('data/processed/sparse/name_char.npz')
    X_dw = load_npz('data/processed/sparse/desc_word.npz')
    X_cb = load_npz('data/processed/sparse/combined.npz')
    X_all = hstack([X_nw, X_nc, X_dw, X_cb]).tocsr()
    del X_nw, X_nc, X_dw, X_cb; gc.collect()
    
    svd = joblib.load('models/svd_v2.joblib')
    X_svd = svd.transform(X_all).astype(np.float32)
    del X_all; gc.collect()
    
    X_numeric = df[numeric_features].values.astype(np.float32)
    X_lgb = np.hstack([X_numeric, X_svd]).astype(np.float32)
    del X_svd, X_numeric; gc.collect()
    print(f"X_lgb: {X_lgb.shape}")

X_tr, X_vl, y_tr, y_vl = train_test_split(X_lgb, y, test_size=0.2, random_state=42)

# LightGBM v2b: farklı parametreler (çeşitlilik için)
lgb_params_b = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.03,     # daha yavaş öğrenme
    'num_leaves': 63,          # daha küçük ağaçlar
    'max_depth': 8,
    'min_data_in_leaf': 200,
    'feature_fraction': 0.7,   # daha az feature
    'bagging_fraction': 0.7,
    'bagging_freq': 3,
    'lambda_l1': 0.1,          # regularization ekle
    'lambda_l2': 0.1,
    'verbose': -1, 'n_jobs': -1, 'seed': 123,  # farklı seed
}

print("LightGBM v2b eğitiliyor (farklı parametreler)...")
t0 = time.time()
tr_ds = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
vl_ds = lgb.Dataset(X_vl, y_vl, reference=tr_ds, free_raw_data=True)

lgb_model_b = lgb.train(
    lgb_params_b, tr_ds, num_boost_round=3000,
    valid_sets=[vl_ds], valid_names=['val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)],
)

print(f"Eğitim: {time.time()-t0:.0f}s")
y_vl_pred_lgb_b = np.maximum(lgb_model_b.predict(X_vl), 0)
lgb_b_rmsle = np.sqrt(mean_squared_error(y_vl, y_vl_pred_lgb_b))
print(f"LightGBM v2b RMSLE: {lgb_b_rmsle:.4f}")

np.save('models/y_val_pred_lgb_v2b.npy', y_vl_pred_lgb_b)

X_lgb bellekte mevcut
LightGBM v2b eğitiliyor (farklı parametreler)...
Training until validation scores don't improve for 50 rounds
[200]	val's rmse: 0.568017
[400]	val's rmse: 0.544146
[600]	val's rmse: 0.532471
[800]	val's rmse: 0.524791
[1000]	val's rmse: 0.519169
[1200]	val's rmse: 0.514783
[1400]	val's rmse: 0.511172
[1600]	val's rmse: 0.508481
[1800]	val's rmse: 0.506217
[2000]	val's rmse: 0.504299
[2200]	val's rmse: 0.502605
[2400]	val's rmse: 0.501132
[2600]	val's rmse: 0.499823
[2800]	val's rmse: 0.498583
[3000]	val's rmse: 0.497591
Did not meet early stopping. Best iteration is:
[3000]	val's rmse: 0.497591
Eğitim: 562s
LightGBM v2b RMSLE: 0.4976


In [8]:
y_val = np.load('models/y_val_v2.npy')
y_ridge = np.load('models/y_val_pred_ridge_v2.npy')
y_lgb_a = np.load('models/y_val_pred_lgb_v2.npy')
y_lgb_b = np.load('models/y_val_pred_lgb_v2b.npy')

ridge_solo = np.sqrt(mean_squared_error(y_val, y_ridge))
lgb_a_solo = np.sqrt(mean_squared_error(y_val, y_lgb_a))
lgb_b_solo = np.sqrt(mean_squared_error(y_val, y_lgb_b))

# 3-model ensemble ağırlık araması
print("3-Model Ensemble Ağırlık Araması")
print("-" * 55)

best_ens3 = float('inf')
best_ws = None

for wr in np.arange(0.5, 0.85, 0.05):
    remaining = 1.0 - wr
    for wa in np.arange(0.05, remaining + 0.01, 0.05):
        wb = round(remaining - wa, 2)
        if wb < 0: continue
        
        ens = wr * y_ridge + wa * y_lgb_a + wb * y_lgb_b
        r = np.sqrt(mean_squared_error(y_val, ens))
        if r < best_ens3:
            best_ens3 = r
            best_ws = (round(wr, 2), round(wa, 2), round(wb, 2))

print(f"\n{'='*55}")
print(f"SONUCLAR")
print(f"{'='*55}")
print(f"  Ridge v2:                 {ridge_solo:.4f}")
print(f"  LightGBM v2a:             {lgb_a_solo:.4f}")
print(f"  LightGBM v2b:             {lgb_b_solo:.4f}")
print(f"  2-model ensemble (eski):  0.4396")
print(f"  3-model ensemble (yeni):  {best_ens3:.4f}")
print(f"  Ağırlık: Ridge={best_ws[0]}, LGBa={best_ws[1]}, LGBb={best_ws[2]}")
print(f"{'='*55}")
print(f"  İyileşme (2→3 model):     {0.4396 - best_ens3:+.4f}")
print(f"  Toplam iyileşme:          {0.4733 - best_ens3:+.4f}")

3-Model Ensemble Ağırlık Araması
-------------------------------------------------------

SONUCLAR
  Ridge v2:                 0.4465
  LightGBM v2a:             0.4881
  LightGBM v2b:             0.4976
  2-model ensemble (eski):  0.4396
  3-model ensemble (yeni):  0.4396
  Ağırlık: Ridge=0.75, LGBa=0.25, LGBb=-0.0
  İyileşme (2→3 model):     -0.0000
  Toplam iyileşme:          +0.0337
